# Results Visualisation

Figures for the condition factorial, in the order the analysis reads them:
instrument checks first, then measurement resolution, then the language results,
then the controls. Definitions and statistics match Notebook 06, which is the
authority for every number shown here; this notebook imports the same functions
from `analysis.py` so the two cannot disagree.

The model is never loaded. Inputs are the prediction log, the harvest manifest, and
the frozen constructed set. Figures are written under the Drive cache as PNGs for
reporting. Isaac Sim arrow figures and rollout videos remain a separate rendering
stage.

Every comparison is read on the axis its spatial term contrasts, and on the
continuous expected-bin readout rather than the quantised action, for the reasons
set out in Notebook 06.

## 1. Mount Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
PRED_CSV = '/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/v2/bridge'
CONSTRUCTED_DIR = '/content/drive/MyDrive/openvla_cache/v2/constructed'
MANIFEST_CSV = os.path.join(CACHE_DIR, 'manifest.csv')
FIGS_DIR = '/content/drive/MyDrive/openvla_cache/v2/results_figs'
os.makedirs(FIGS_DIR, exist_ok=True)
print('predictions ->', PRED_CSV)
print('constructed ->', CONSTRUCTED_DIR)
print('figures     ->', FIGS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
predictions -> /content/drive/MyDrive/openvla_cache/probe_predictions_v4.csv
constructed -> /content/drive/MyDrive/openvla_cache/constructed
figures     -> /content/drive/MyDrive/openvla_cache/results_figs


## 2. Import the code

Clones the project code from GitHub into the runtime and imports the shared
analysis functions, so the figures always match the pushed commit and the
statistics reported in Notebook 06.

In [5]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'data', 'controls', 'compose_scenes', 'export_pairs', 'analysis',):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

import analysis
from analysis import (axis_value, wilcoxon_paired, resolution_report,
                      mirror_check, lexical_check, term_effect,
                      congruence_test, absolute_congruence, same_side_test,
                      CONTINUOUS_COLS)
from export_pairs import load_inputs, build_pairs
from compose_scenes import (evaluation_scenes, load_constructed_manifest,
                            IMAGE_X_TO_LATERAL_SIGN)
from data import AXIS_INDEX
print(f'imported project modules from {module_dir} @ {commit}')

imported project modules from /content/ECS8056 @ bb17c75


## 3. Load and prepare

`load_inputs` validates that the prediction log carries the required probe
columns and fails loudly if any are missing. Rows without a continuous readout
were migrated from an earlier log and are held back from the continuous figures.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Lateral bin width from the model's decoding constants, printed by Notebook 05.
BIN_WIDTH = 0.0011180

# One consistent palette across every figure.
COLOURS = {
    'constructed': '#4C72B0', 'bridge': '#DD8452',
    'opposite': '#4C72B0', 'same_side_left': '#55A868', 'same_side_right': '#8172B3',
    'baseline': '#4C72B0', 'swapped_scene': '#C44E52',
    'argmax': '#937860', 'continuous': '#4C72B0',
}
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

preds, manifest = load_inputs(PRED_CSV, MANIFEST_CSV)
coverage = preds[CONTINUOUS_COLS].notna().all(axis=1)
usable = preds[coverage].copy()
usable['value'] = axis_value(usable)
print(f'{len(preds)} predictions | continuous readout on {coverage.sum()} '
      f'({coverage.mean():.1%})')

def save(fig, name):
    out = os.path.join(FIGS_DIR, name)
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print('wrote', out)
    return out

def paired_difference(frame, condition='baseline', continuous=True):
    """Signed A minus B difference per scene on that scene's own axis."""
    sub = frame[frame['condition'] == condition].copy()
    if sub.empty:
        return pd.DataFrame(columns=['scene_id', 'diff'])
    sub['value'] = axis_value(sub, continuous=continuous)
    wide = sub.pivot_table(index='scene_id', columns='role', values='value')
    if not {'a', 'b'} <= set(wide.columns):
        return pd.DataFrame(columns=['scene_id', 'diff'])
    wide = wide.dropna(subset=['a', 'b'])
    return pd.DataFrame({'scene_id': wide.index,
                         'diff': (wide['a'] - wide['b']).to_numpy()})

ValueError: /content/drive/MyDrive/openvla_cache/probe_predictions_v4.csv is not readable as a table: its header declares 34 columns but its rows have field counts {34: 2202, 66: 516}. This happens when a log is appended to after its schema widened. Repair it with prediction_log.repair_log, passing the current schema from prediction_log.canonical_log_fields (the 'Check the log is readable' cell of notebook 03 does this).

## 4. Inventory

What was actually collected. Unequal counts per condition are expected: a
condition is skipped when its precondition fails, for example a mirror condition
on a depth term, or a term-stripped condition where no clean removal exists.
Every figure below is conditional on these counts.

In [ ]:
inventory = pd.crosstab(usable['condition'], usable['scene_source'])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

inventory.plot(kind='barh', ax=axes[0],
               color=[COLOURS.get(c, '#7F7F7F') for c in inventory.columns])
axes[0].set_xlabel('Predictions')
axes[0].set_ylabel('')
axes[0].set_title('Predictions by condition and scene source')
axes[0].legend(frameon=False, fontsize=8)

built = usable[usable['scene_source'] == 'constructed']
if not built.empty:
    counts = built.groupby('configuration')['scene_id'].nunique().sort_values()
    axes[1].barh(counts.index.astype(str), counts.to_numpy(),
                 color=[COLOURS.get(c, '#7F7F7F') for c in counts.index])
    axes[1].set_xlabel('Scenes')
    axes[1].set_title('Constructed scenes by configuration')
else:
    axes[1].text(0.5, 0.5, 'No constructed scenes', ha='center', va='center')
    axes[1].axis('off')

fig.tight_layout()
save(fig, '01_inventory.png')
plt.show()
print(inventory.to_string())

## 5. Instrument check: response to mirroring

Each point is one scene: its lateral value on the original image against its
value on the horizontally flipped image, with the instruction held fixed. A model
that reads lateral position must reverse sign, placing points on the falling
anti-diagonal. A model that ignores the image leaves them on the rising diagonal.

The left panel uses the term-stripped instruction and is the cleaner reading,
since it isolates object grounding from any influence of the spatial word. This
figure is the gate: if the points sit on the rising diagonal, the visual channel
is not live on this axis and no language conclusion can be drawn from the figures
that follow.

The constructed arrangements are separate series for a reason internal to the
construction. Reflecting an `opposite` arrangement maps the layout onto itself, so
those points sit near the origin however the model behaves; the same-side series
is where the reflection genuinely moves the scene, and it is the one to read.

In [ ]:
def mirror_pairs(frame, plain, flipped, role):
    """Per-scene lateral value on the original and mirrored image."""
    out = {}
    for label, condition in (('plain', plain), ('mirror', flipped)):
        sub = frame[(frame['condition'] == condition) & (frame['role'] == role)].copy()
        if sub.empty:
            return pd.DataFrame(columns=['plain', 'mirror'])
        sub['value'] = axis_value(sub)
        out[label] = sub.groupby('scene_id')['value'].mean()
    return pd.DataFrame(out).dropna()

lateral = usable[usable['axis_index'] == AXIS_INDEX['lateral']]
panels = [('term stripped', 'neutral', 'mirror_neutral', 'n'),
          ('spatial term present', 'baseline', 'mirror', 'a')]

# The constructed arrangements are drawn as separate series. Reflecting an
# opposite arrangement maps the layout onto itself, so those points cluster near
# the origin whatever the model does, and pooling them with the same-side
# arrangements would obscure the only stratum the figure can read.
constructed = lateral[lateral['scene_source'] == 'constructed']
same_side = constructed['configuration'].astype(str).str.startswith('same_side')
series = [
    ('constructed, same side', constructed[same_side], COLOURS['same_side_left']),
    ('constructed, opposite', constructed[~same_side], COLOURS['opposite']),
    ('bridge', lateral[lateral['scene_source'] == 'bridge'], COLOURS['bridge']),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (title, plain, flipped, role) in zip(axes, panels):
    drawn = False
    for label, frame, colour in series:
        points = mirror_pairs(frame, plain, flipped, role)
        if points.empty:
            continue
        ax.scatter(points['plain'], points['mirror'], s=22, alpha=0.6,
                   color=colour, label=f'{label} (n={len(points)})')
        drawn = True
    if not drawn:
        ax.text(0.5, 0.5, 'no paired mirror predictions', ha='center', va='center')
        ax.axis('off')
        continue
    span = max(abs(np.array(ax.get_xlim())).max(), abs(np.array(ax.get_ylim())).max())
    ax.plot([-span, span], [span, -span], color='#55A868', lw=1.2,
            label='reverses sign (grounded)')
    ax.plot([-span, span], [-span, span], color='#C44E52', ls='--', lw=1.2,
            label='unchanged (ignores image)')
    ax.axhline(0, color='grey', lw=0.6)
    ax.axvline(0, color='grey', lw=0.6)
    ax.set_xlabel('lateral value, original image')
    ax.set_ylabel('lateral value, mirrored image')
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)

fig.suptitle('Does the lateral output respond to reflecting the scene?')
fig.tight_layout()
save(fig, '02_instrument_mirror.png')
plt.show()

## 6. Measurement resolution

What the quantisation was hiding. The same paired differences are shown twice:
as the argmax action the model would execute, and as the continuous expected-bin
value. The shaded band is one action bin, the width below which two predictions
cannot differ in the executable action.

The earlier analysis reported a median paired difference of exactly zero in every
stratum. The left panel is that result; the right panel is the same data with the
resolution the argmax discards.

In [ ]:
# Collected first so both panels share one scale: the comparison between the two
# readouts is the point of the figure, and it is lost if each is scaled to its
# own spread. An explicit limit is also needed because a degenerate
# distribution, which the argmax panel may well be, would otherwise be
# auto-scaled to an arbitrary default range.
series = {}
for readout, flag in (('argmax', False), ('continuous', True)):
    for source in ('constructed', 'bridge'):
        diffs = paired_difference(usable[usable['scene_source'] == source],
                                  continuous=flag)['diff'].to_numpy()
        if diffs.size:
            series[(readout, source)] = diffs

widest = max((np.abs(d).max() for d in series.values()), default=BIN_WIDTH)
limit = max(3 * BIN_WIDTH, 1.1 * widest)
edges = np.linspace(-limit, limit, 61)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True, sharex=True)
rows = []
for ax, readout in zip(axes, ('argmax', 'continuous')):
    drawn = False
    for source in ('constructed', 'bridge'):
        diffs = series.get((readout, source))
        if diffs is None:
            continue
        ax.hist(diffs, bins=edges, alpha=0.6, color=COLOURS[source],
                label=f'{source} (n={diffs.size})')
        rows.append({'scene_source': source, 'readout': readout,
                     **resolution_report(diffs, BIN_WIDTH)})
        drawn = True
    ax.axvspan(-BIN_WIDTH, BIN_WIDTH, color='grey', alpha=0.18,
               label='one action bin')
    ax.axvline(0, color='grey', lw=0.8)
    ax.set_xlim(-limit, limit)
    ax.set_xlabel('paired difference (A minus B)')
    ax.set_title(f'{readout} readout')
    if drawn:
        ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel('Pairs')
fig.suptitle('Paired difference before and after clearing the quantisation floor')
fig.tight_layout()
save(fig, '03_resolution.png')
plt.show()

resolution = pd.DataFrame(rows)
if not resolution.empty:
    print(resolution[['scene_source', 'readout', 'n', 'frac_exact_zero',
                      'frac_below_bin', 'n_distinct']].to_string(index=False))

## 7. The decisive comparison

On a scene with both instances on the same side of the start position, scene
grounding and a word-to-direction mapping predict opposite things. A grounded
model selects between two targets lying in the same direction, so both
instructions produce a same-signed action. A model that maps `left` to one
direction and `right` to the other produces oppositely-signed actions.

On the `opposite` configuration the two accounts agree, which is why the original
sign-flip metric could not tell them apart. The gap between the bars is the
result; the height of the `opposite` bar alone is not.

The right panel states the same discrimination directly, scoring each instruction
against the side its own target occupies rather than scoring the pair against
their relative order. On the same-side bars, near one is scene grounding and near
one half is a word-to-direction mapping, which can only ever point at one of two
targets that share a side. Near zero is scene grounding read under an inverted
sign convention rather than a third account.

Both panels count only predictions of at least one action bin, since a smaller
value decodes to the same executable action whichever sign it carries. The title
quotes the contrast paired within the base frame, which compares the two
arrangements built from the same frame and so holds everything but the arrangement
fixed.

In [ ]:
if built.empty:
    print('no constructed predictions; run Notebooks 03 and 05 first')
else:
    # Held to one action bin, matching the figure the analysis notebook quotes:
    # a sub-bin sign is not a decision the model could execute.
    result = same_side_test(built, min_magnitude=BIN_WIDTH)
    entries = sorted(result['by_configuration'].items())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    names = [n for n, _ in entries]
    rates = [e['same_sign_rate'] for _, e in entries]
    axes[0].bar(names, rates, color=[COLOURS.get(n, '#7F7F7F') for n in names])
    for i, (name, entry) in enumerate(entries):
        if np.isfinite(entry['same_sign_rate']):
            axes[0].text(i, entry['same_sign_rate'] + 0.02, f"n={entry['n']}",
                         ha='center', fontsize=9)
    axes[0].axhline(0.5, color='grey', ls='--', lw=0.8)
    axes[0].set_ylim(0, 1.12)
    axes[0].set_ylabel('Both instructions same-signed')
    axes[0].set_title('Grounded: high on same-side, low on opposite')
    axes[0].tick_params(axis='x', labelrotation=15)

    # Each instruction scored against the side its own target occupies. On a
    # same-side arrangement a grounded model reaches one, and a fixed
    # word-to-direction mapping can only ever satisfy one of the two.
    absolute = absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN,
                                   min_magnitude=BIN_WIDTH)
    entries = sorted((n, e) for n, e in absolute.get('by_configuration', {}).items()
                     if e.get('n'))
    if entries:
        labels = [n for n, _ in entries]
        axes[1].bar(labels, [e['agreement'] for _, e in entries],
                    color=[COLOURS.get(n, '#7F7F7F') for n in labels])
        for i, (_, entry) in enumerate(entries):
            axes[1].text(i, entry['agreement'] + 0.02, f"n={entry['n']}",
                         ha='center', fontsize=9)
        axes[1].axhline(0.5, color='grey', ls='--', lw=0.8)
        axes[1].text(0.99, 0.505, 'word-to-direction mapping', fontsize=8,
                     color='grey', ha='right', va='bottom',
                     transform=axes[1].get_yaxis_transform())
        axes[1].set_ylim(0, 1.12)
        axes[1].set_ylabel('Instructions moving toward their own target')
        axes[1].set_title('Does each instruction reach its named target?')
        axes[1].tick_params(axis='x', labelrotation=15)
    else:
        axes[1].text(0.5, 0.5, 'target sides were not logged;\nre-run Notebook 05',
                     ha='center', va='center')
        axes[1].axis('off')

    # The paired figure where the frozen set supplies both arrangements of a
    # frame, since that is the contrast with nothing else varying between the
    # two sides of the comparison.
    paired = result['contrast_paired']
    contrast = (paired if paired.get('n_pairs') else result['contrast'])
    scope = (f"paired within {paired['n_pairs']} base frames"
             if paired.get('n_pairs') else 'unpaired')
    fig.suptitle(f"same-side minus opposite = {contrast['difference']:+.1%} "
                 f"(p={contrast['p_value']:.3g}, {scope})")
    fig.tight_layout()
    save(fig, '04_same_side_contrast.png')
    plt.show()

## 8. Controls

Left: the instruction contrast run against the scene it describes, and against a
scene it does not. A swapped-scene distribution as wide as the baseline means the
response is a property of the words rather than of the scene.

Right: each instruction as a deviation from the prediction on the identical image
with the spatial term removed. Points in the upper-left and lower-right quadrants
are scenes where the two variants moved in opposite directions from their shared
reference, which is what a directional term should produce.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

for condition in ('baseline', 'swapped_scene'):
    diffs = paired_difference(usable, condition=condition)['diff'].to_numpy()
    if diffs.size == 0:
        continue
    axes[0].hist(diffs, bins=30, alpha=0.6, color=COLOURS[condition],
                 label=f'{condition} (n={diffs.size})')
axes[0].axvspan(-BIN_WIDTH, BIN_WIDTH, color='grey', alpha=0.18,
                label='one action bin')
axes[0].axvline(0, color='grey', lw=0.8)
axes[0].set_xlabel('paired difference (A minus B)')
axes[0].set_ylabel('Pairs')
axes[0].set_title('Does the contrast survive without its scene?')
axes[0].legend(frameon=False, fontsize=8)

def deviations(frame):
    """Per-scene deviation of each variant from the term-stripped reference."""
    wide = {}
    for condition, role, key in (('baseline', 'a', 'a'), ('baseline', 'b', 'b'),
                                 ('neutral', 'n', 'n')):
        sub = frame[(frame['condition'] == condition) & (frame['role'] == role)].copy()
        if sub.empty:
            return pd.DataFrame(columns=['a', 'b', 'n'])
        sub['value'] = axis_value(sub)
        wide[key] = sub.groupby('scene_id')['value'].mean()
    table = pd.DataFrame(wide).dropna()
    return pd.DataFrame({'dev_a': table['a'] - table['n'],
                         'dev_b': table['b'] - table['n']})

drawn = False
for source in ('constructed', 'bridge'):
    dev = deviations(usable[usable['scene_source'] == source])
    if dev.empty:
        continue
    axes[1].scatter(dev['dev_a'], dev['dev_b'], s=22, alpha=0.6,
                    color=COLOURS[source], label=f'{source} (n={len(dev)})')
    drawn = True
if drawn:
    span = max(abs(np.array(axes[1].get_xlim())).max(),
               abs(np.array(axes[1].get_ylim())).max())
    axes[1].plot([-span, span], [span, -span], color='#55A868', lw=1.2,
                 label='opposite deviations')
    axes[1].axhline(0, color='grey', lw=0.6)
    axes[1].axvline(0, color='grey', lw=0.6)
    axes[1].legend(frameon=False, fontsize=8)
else:
    axes[1].text(0.5, 0.5, 'no scenes with both a pair and a neutral reference',
                 ha='center', va='center')
axes[1].set_xlabel('deviation of variant A from the term-free reference')
axes[1].set_ylabel('deviation of variant B')
axes[1].set_title("The spatial term's marginal effect")

fig.tight_layout()
save(fig, '05_controls.png')
plt.show()

for source in sorted(usable['scene_source'].unique()):
    result = lexical_check(usable[usable['scene_source'] == source])
    if np.isfinite(result['ratio']):
        print(f'{source:12} swapped/baseline ratio = {result["ratio"]:.2f}')

## 9. Validation: unaltered BridgeData strata

The real, unmodified frames, by stratum, held disjoint from the base frames the
constructed stimuli were built on. They test whether the result above survives
outside the composited set, being the distribution the model was trained on with no
edited pixels. They cannot carry the experiments themselves: the well-posed subset
is small, because a frame holding one instance of the target lets the model succeed
by finding the only candidate and such frames are rare in the corpus.

Counts are shown beside each bar because the well-posed stratum is small enough
that its estimate is fragile, and a bar height without its sample size would
overstate what the figure supports.

In [ ]:
bridge = usable[usable['scene_source'] == 'bridge']
if bridge.empty:
    print('no bridge predictions in the log')
else:
    strata = {
        'well-posed\n(feasible, duplicate target)':
            bridge[(bridge['category'] == 'referent_selection')
                   & (bridge['feasible_both'] == 'yes')
                   & (bridge['duplicate_target'] == 'yes')],
        'referent_selection\nfeasible':
            bridge[(bridge['category'] == 'referent_selection')
                   & (bridge['feasible_both'] == 'yes')],
        'all bridge\nscenes': bridge,
    }
    labels, medians, counts, zeros = [], [], [], []
    for label, frame in strata.items():
        diffs = paired_difference(frame)['diff'].to_numpy()
        labels.append(label)
        medians.append(float(np.median(diffs)) if diffs.size else 0.0)
        counts.append(diffs.size)
        zeros.append(float(np.mean(diffs == 0)) if diffs.size else np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].bar(labels, medians, color=COLOURS['bridge'])
    for i, (m, n) in enumerate(zip(medians, counts)):
        axes[0].text(i, m, f'  n={n}', ha='center', va='bottom', fontsize=9)
    axes[0].axhline(0, color='grey', lw=0.8)
    axes[0].axhspan(-BIN_WIDTH, BIN_WIDTH, color='grey', alpha=0.18,
                    label='one action bin')
    axes[0].set_ylabel('median paired difference')
    axes[0].set_title('Median difference by stratum')
    axes[0].legend(frameon=False, fontsize=8)
    axes[0].tick_params(axis='x', labelsize=8)

    axes[1].bar(labels, zeros, color='#937860')
    axes[1].set_ylim(0, 1.05)
    axes[1].set_ylabel('fraction exactly zero')
    axes[1].set_title('How much of each stratum is at the floor')
    axes[1].tick_params(axis='x', labelsize=8)

    fig.tight_layout()
    save(fig, '06_bridge_strata.png')
    plt.show()

## 10. Constructed scene gallery

Examples of the stimuli, with the recorded geometry drawn on: the source
instance, the pasted duplicate, and the start position. Included so a reader can
see what the model was shown and judge whether the composited instance is
plausible, which no summary statistic conveys.

In [ ]:
from PIL import Image

GALLERY_N = 4
# The frozen set, so the gallery shows stimuli the experiments actually used.
scenes = evaluation_scenes(CONSTRUCTED_DIR)
if not scenes:
    print('no frozen evaluation set; screen and freeze in Notebook 03 first')
else:
    # One example per configuration, then fill up to GALLERY_N.
    chosen, seen = [], set()
    for scene in scenes:
        if scene['configuration'] not in seen:
            chosen.append(scene)
            seen.add(scene['configuration'])
    chosen += [s for s in scenes if s not in chosen][:max(0, GALLERY_N - len(chosen))]
    chosen = chosen[:GALLERY_N]

    if chosen:
        fig, axes = plt.subplots(1, len(chosen), figsize=(4 * len(chosen), 4.6))
        axes = [axes] if len(chosen) == 1 else list(axes)
        for ax, scene in zip(axes, chosen):
            image = Image.open(os.path.join(CONSTRUCTED_DIR, scene['image_path']))
            ax.imshow(image)
            y = image.height * 0.5
            ax.scatter([float(scene['x_source'])], [y], marker='o', s=110,
                       facecolors='none', edgecolors='#4C72B0', linewidths=2.2,
                       label='source instance')
            ax.scatter([float(scene['x_pasted'])], [y], marker='s', s=110,
                       facecolors='none', edgecolors='#DD8452', linewidths=2.2,
                       label='pasted instance')
            ax.axvline(float(scene['x_gripper']), color='#55A868', ls='--',
                       lw=1.6, label='start position')
            ax.set_title(f"{scene['configuration']}\n{scene['instr_a']}", fontsize=8)
            ax.axis('off')
        axes[0].legend(loc='lower left', fontsize=7)
        fig.tight_layout()
        save(fig, '07_constructed_gallery.png')
        plt.show()

## 11. Figure index

All PNGs written in this session under `results_figs/`.

In [ ]:
written = sorted(f for f in os.listdir(FIGS_DIR) if f.endswith('.png'))
print(f'{len(written)} figures in {FIGS_DIR}:')
for name in written:
    print(' ', name)